An example of using a defined footprint area (here, pulled from the scheduler definitions of different areas) to calculate a metric.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import sqlite3

import rubin_sim.maf as maf
from rubin_scheduler.scheduler.utils import SkyAreaGenerator, EuclidOverlapFootprint
from rubin_sim.data import get_baseline

In [ ]:
# Pick up the "footprint of interest"
# The current survey footprint is defined in EuclidOverlapFootprint

nside = 64
surveyAreas = EuclidOverlapFootprint(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()

In [ ]:
# What areas are labelled in the map?
np.unique(map_labels)

In [ ]:
# Let's say the "area of interest" is the low-dust-extinction WFD
# In practice, this likely includes the euclid_overlap and virgo areas which match lowdust in filter balance and visits
# (note that the bulge and LMC_SMC are also WFD-level in total # of visits)

lowdust = np.where(
    (map_labels == "lowdust") | (map_labels == "euclid_overlap") | (map_labels == "virgo"), 1, 0
)
hp.mollview(lowdust)

In [ ]:
# Let's just connect to the opsim output and check what "notes" are available too
# We might want to reject or only include some subsets of visits due to survey mode

opsdb = get_baseline()
run_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb, "--", run_name)

In [ ]:
conn = sqlite3.connect(opsdb)
d = pd.read_sql("select distinct(note) from observations", conn)
conn.close()
d

In [ ]:
# Let's use these healpixels to calculate our metric -- using a HealpixSubsetSlicer
# Reject DDF and twilight near-sun visits (these are only 15s)

count_metric = maf.CountMetric(col="observationStartMJD", metric_name="NVisits")
depth_metric = maf.Coaddm5Metric()
slicer = maf.HealpixSubsetSlicer(nside=nside, hpid=np.where(map_labels == "lowdust")[0])
constraint = 'note not like "DD%" and note not like "%twi%"'

count_bundle = maf.MetricBundle(count_metric, slicer, constraint, run_name=run_name)
depth_bundle = maf.MetricBundle(depth_metric, slicer, constraint + " and filter == 'r'", run_name=run_name)

In [ ]:
g = maf.MetricBundleGroup(
    {"count": count_bundle, "depth": depth_bundle}, opsdb, out_dir="tmp_out", verbose=True
)

In [ ]:
g.run_all()

In [ ]:
count_bundle.plot()

In [ ]:
depth_bundle.plot()

In [ ]:
count_bundle.set_summary_metrics(maf.extended_summary())
depth_bundle.set_summary_metrics(maf.extended_summary())

count_bundle.compute_summary_stats()
depth_bundle.compute_summary_stats()

In [ ]:
pd.DataFrame([count_bundle.summary_values, depth_bundle.summary_values], index=["Count", "r band depth"])